In [ ]:
import json
from pathlib import Path

from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder

In [ ]:
class argsparser:
    def __init__(self):
        self.docs_dir = "../data"
        self.chroma_path = "./chroma_db3"
        self.collection = "documents"
        self.chunk_tokens = 512
        self.embed_model = "sentence-transformers/all-MiniLM-L6-v2"
        self.save_text_out = None
        self.batch_size = 64

args = argsparser()

In [ ]:
docs_dir = Path(args.docs_dir)
if not docs_dir.exists():
    raise FileNotFoundError(f"Folder not found: {docs_dir}")

pdf_files = sorted(docs_dir.glob("*.pdf"))
if not pdf_files:
    raise SystemExit(f"No PDFs found in {docs_dir}")

print(f"Found {len(pdf_files)} PDF(s) in {docs_dir}")

In [ ]:
pdf_files = pdf_files[:5]

In [ ]:
text_out_dir = Path(args.save_text_out) if args.save_text_out else None
if text_out_dir:
    text_out_dir.mkdir(parents=True, exist_ok=True)

print(f"Loading embedding model: {args.embed_model}")
embedder = SentenceTransformer(args.embed_model)

print(f"Opening ChromaDB at: {args.chroma_path}")
client = chromadb.PersistentClient(path=args.chroma_path)
collection = client.get_or_create_collection(name=args.collection)

In [ ]:
converter = DocumentConverter() # no image pipeline options - text/table only, faster
chunker = HybridChunker(tokenizer=args.embed_model, max_tokens=args.chunk_tokens)

total_chunks = 0
failed_files = []

for pdf_idx, pdf_path in enumerate(pdf_files, 1):
    print(f"\n[{pdf_idx}/{len(pdf_files)}] {pdf_path.name}")
    try:
        result = converter.convert(str(pdf_path))
        doc = result.document
    except Exception as e:
        print(f"Failed to convert: {e}")
        failed_files.append((pdf_path.name, str(e)))
        continue
        
    chunks = list(chunker.chunk(doc))
    if not chunks:
        print("No chunks produced (empty or unparseable PDF) - skipping.")
        continue
        
    ids, texts, metadatas = [], [], []
    for i, chunk in enumerate(chunks):
        doc_items = chunk.meta.doc_items or []
        page_numbers = sorted({prov.page_no for item in doc_items for prov in item.prov})
        headings = getattr(chunk.meta, "headings", None) or []
        
        text = chunker.contextualize(chunk)
        ids.append(f"{pdf_path.stem}-chunk-{i}")
        texts.append(text)
        
        metadatas.append({
            "source_file": pdf_path.name,
            "chunk_index": i,
            "page_numbers": ",".join(str(p) for p in page_numbers) if page_numbers else "",
            "headings": " > ".join(headings) if headings else "",
        })
        
    # Embed in batches to keep memory reasonable on large PDFs
    embeddings = []
    for start in range(0, len(texts), args.batch_size):
        batch = texts[start:start + args.batch_size]
        embeddings.extend(embedder.encode(batch, show_progress_bar=False).tolist())
        
    collection.add(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
    total_chunks += len(chunks)
    print(f"Added {len(chunks)} chunks to collection ({args.collection})")
    
    if text_out_dir:
        (text_out_dir / f"{pdf_path.stem}.md").write_text(doc.export_to_markdown(), encoding="utf-8")
        json_path = text_out_dir / f"{pdf_path.stem}.chunks.json"
        with json_path.open("w", encoding="utf-8") as f:
            for _id, _text, _meta in zip(ids, texts, metadatas):
                f.write(json.dumps({"id": _id, "text": _text, "meta": _meta}, ensure_ascii=False) + "\n")

print(f"\n{'='*50}")
print(f"Done. Total {total_chunks} chunks from {len(pdf_files) - len(failed_files)}/{len(pdf_files)} PDFs stored in ChromaDB collection ({args.collection}) at ({args.chroma_path})")

if failed_files:
    print(f"\nFailed {len(failed_files)} file(s) failed:")
    for name, err in failed_files:
        print(f"- {name}: {err}")

print(f"\nCollection now has {collection.count()} total chunks.")

In [ ]:
all_items = collection.get(include=["documents", "metadatas"])
ids = all_items.get("ids", [])
documents = all_items.get("documents", [])
metadatas = all_items.get("metadatas", [])

In [ ]:
metadatas[0]

## Extract Metadata

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

DOCUMENT_METADATA_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a document metadata extraction assistant for a company
documentation RAG system.

Analyze the supplied document text and create document-level metadata.

Rules:
1. Generate a concise ONE-LINE summary explaining what the document is about.
2. The summary must be factual and based only on the supplied text.
3. Keep the summary concise, normally 10–30 words.
4. Extract important entities explicitly supported by the document.
5. Entities may include locations, organizations, departments, policies,
   systems, dates, business areas, and subjects.
6. Identify the document type.
7. Identify the primary location only if explicitly supported.
8. Identify the main topics.
9. Do NOT invent information.
10. If a field cannot be determined reliably, return an empty string or empty list.
11. Do not answer a user question; only produce document metadata.
12. document_category MUST be exactly one of:
    holiday, policy, procedure, faq, technical, report, manual, training, other.
13. Use "holiday" for holiday calendars/schedules.
14. Use "faq" for frequently asked questions or FAQ documents.
15. Use "procedure" for process/procedure/how-to documents.
16. Use "policy" for policy documents.
17. Use "technical" for technical/reference/system documentation.
18. Use "report" for reports/analysis documents.
19. Use "manual" for user manuals/instructions.
20. Use "training" for training/learning material.
21. Use "other" only when no category above is suitable.
""",
    ),
    (
        "human",
        "Document text:\\n{document_text}",
    ),
])

from pydantic import BaseModel, Field

class DocumentMetadata(BaseModel):
    """Structured metadata generated for each source document."""

    summary: str = Field(
        description=(
            "A concise one-line summary explaining what the document is about. "
            "Keep it between 10 and 30 words."
        )
    )
    entities: list[str] = Field(
        default_factory=list,
        description=(
            "Important entities explicitly supported by the document, such as "
            "locations, organizations, departments, policies, systems, dates, "
            "business areas, or document subjects."
        ),
    )
    document_type: str = Field(
        default="",
        description=(
            "The type of document, for example Holiday Calendar, Policy, "
            "Procedure, Technical Document, Report, or Manual."
        ),
    )
    document_category: str = Field(
        default="other",
        description=(
            "A normalized document category. Use one of: holiday, policy, "
            "procedure, faq, technical, report, manual, training, other."
        ),
    )
    location: str = Field(
        default="",
        description=(
            "Primary location explicitly associated with the document, if one "
            "exists. Otherwise return an empty string."
        ),
    )
    topics: list[str] = Field(
        default_factory=list,
        description="The main topics covered by the document.",
    )

In [ ]:
from langchain_groq import ChatGroq
import os
try:
    from google.colab import userdata
    key = userdata.get("GROQ_API_KEY")
except Exception:
    key = os.environ.get("GROQ_API_KEY")

groq_model = "openai/gpt-oss-20b"

llm = ChatGroq(
    model=groq_model,
    api_key=key,
    temperature=0,
    max_tokens=1200,
)

document_metadata_chain = (
    DOCUMENT_METADATA_PROMPT
    | llm.with_structured_output(DocumentMetadata)
)

In [ ]:
all_items = collection.get(include=["documents", "metadatas"])

ids = all_items.get("ids", [])
documents = all_items.get("documents", [])
metadatas = all_items.get("metadatas", [])

In [ ]:
text_parts = {}

for doc, meta in zip(documents, metadatas):
    meta = meta or {}
    source = meta.get("source_file", "unknown")
    text_parts.setdefault(source, []).append(doc or "")

document_text = "\\n\\n".join(text_parts["CARD ACTIVATION PROCESS.pdf"])

In [ ]:
source = sample_source if "sample_source" in locals() else "CARD ACTIVATION PROCESS.pdf"
print(f"Generating metadata for: {source}")

if document_text:
    result = document_metadata_chain.invoke(
        {"document_text": document_text}
    )

print("####################")
print("Summary:", result.summary)
print("Entities:", result.entities)
print("Document Type:", result.document_type)
print("Document Category:", result.document_category)
print("Location:", result.location)
print("Topics:", result.topics)

In [ ]:
import re
import hashlib

def _metadata_fingerprint(source, texts) -> str:
    """Stable fingerprint for a source file and all chunk content."""
    payload = str(source or "") + "\\n" + "\\n".join(
        str(t or "") for t in texts
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def _metadata_is_complete(meta) -> bool:
    """Return True when all metadata required for retrieval exists."""
    required_fields = [
        "summary",
        "entities",
        "document_type",
        "document_category",
        "location",
        "topics",
    ]
    return all(str(meta.get(field, "") or "").strip() for field in required_fields)


def infer_location_from_source(source_file) -> str:
    """Conservative filename fallback."""
    # Example: Holiday Calendar 2025 - Bangalore.pdf -> Bangalore
    if not source_file:
        return ""

    filename = str(source_file).split("/")[-1].strip()
    match = re.search(
        r"[-–—]\\s*([A-Z][A-Za-z .&'-]*)\\.pdf$",
        filename,
        re.IGNORECASE,
    )
    return match.group(1).strip() if match else ""


def _normalise_metadata_list(values) -> str:
    if not values:
        return ""
    cleaned = []
    for value in values:
        value = str(value).strip()
        if value and value not in cleaned:
            cleaned.append(value)
    return ", ".join(cleaned)

In [ ]:
def enrich_collection_with_document_metadata(
    collection,
    document_metadata_chain,
    max_chars_per_document: int = 12000,
) -> dict:
    """
    Incrementally generate and persist document-level metadata.

    Existing complete metadata is reused. The LLM is called only when metadata
    is missing/incomplete. A content fingerprint is stored on each chunk so
    future runs can detect content changes.
    """
    all_items = collection.get(include=["documents", "metadatas"])
    ids = all_items.get("ids", [])
    documents = all_items.get("documents", [])
    metadatas = all_items.get("metadatas", [])

    if not ids:
        return {}

    grouped = {}
    for item_id, doc, meta in zip(ids, documents, metadatas):
        meta = meta or {}
        source = meta.get("source_file", "unknown")
        grouped.setdefault(
            source, {"ids": [], "texts": [], "metadatas": []}
        )
        grouped[source]["ids"].append(item_id)
        grouped[source]["texts"].append(doc or "")
        grouped[source]["metadatas"].append(meta)

    generated = {}
    reused_count = 0
    generated_count = 0

    print("\\nChecking document-level metadata...")

    for source, data in grouped.items():
        fingerprint = _metadata_fingerprint(source, data["texts"])
        cached = None

        # First preference: exact fingerprint + complete metadata.
        for meta in data["metadatas"]:
            if (
                str(meta.get("metadata_fingerprint", "") or "").strip()
                == fingerprint
                and _metadata_is_complete(meta)
            ):
                cached = {
                    "summary": str(meta.get("summary", "") or "").strip(),
                    "entities": str(meta.get("entities", "") or "").strip(),
                    "document_type": str(meta.get("document_type", "") or "").strip(),
                    "document_category": str(meta.get("document_category", "") or "").strip(),
                    "location": str(meta.get("location", "") or "").strip(),
                    "topics": str(meta.get("topics", "") or "").strip(),
                }
                break

        # Backward compatibility: reuse complete metadata created before fingerprints.
        if cached is None:
            for meta in data["metadatas"]:
                if _metadata_is_complete(meta):
                    cached = {
                        "summary": str(meta.get("summary", "") or "").strip(),
                        "entities": str(meta.get("entities", "") or "").strip(),
                        "document_type": str(meta.get("document_type", "") or "").strip(),
                        "document_category": str(meta.get("document_category", "") or "").strip(),
                        "location": str(meta.get("location", "") or "").strip(),
                        "topics": str(meta.get("topics", "") or "").strip(),
                    }
                    break

        if cached is not None:
            if not cached["location"]:
                cached["location"] = infer_location_from_source(source)
            generated[source] = cached
            reused_count += 1
            print(f"Using cached metadata: {source}")
            for meta in data["metadatas"]:
                meta["metadata_fingerprint"] = fingerprint
            continue

        # Metadata is genuinely missing/incomplete.
        text_parts = []
        current_length = 0

        for chunk_text in data["texts"]:
            chunk_text = str(chunk_text or "").strip()
            if not chunk_text:
                continue

            remaining = max_chars_per_document - current_length
            if remaining <= 0:
                break

            selected = chunk_text[:remaining]
            text_parts.append(selected)
            current_length += len(selected)

        document_text = "\\n\\n".join(text_parts)
        print(f"Generating metadata for: {source}")

        try:
            if document_text:
                result = document_metadata_chain.invoke(
                    {"document_text": document_text}
                )
                summary = str(result.summary or "").strip()
                entities = _normalise_metadata_list(result.entities)
                document_type = str(result.document_type or "").strip()
                document_category = str(result.document_category or "").strip()
                location = str(result.location or "").strip()
                topics = _normalise_metadata_list(result.topics)
            else:
                summary = entities = document_type = location = topics = ""
                document_category = ""

            if not location:
                location = infer_location_from_source(source)

            generated[source] = {
                "summary": summary,
                "entities": entities,
                "document_type": document_type,
                "document_category": document_category,
                "location": location,
                "topics": topics,
            }
            generated_count += 1

        except Exception as e:
            print(f"Warning: could not generate metadata for {source}: {e}")
            generated[source] = {
                "summary": "",
                "entities": "",
                "document_type": "",
                "document_category": "",
                "location": infer_location_from_source(source),
                "topics": "",
            }

    # Persist metadata and fingerprint to every chunk.
    for source, metadata in generated.items():
        source_ids = grouped[source]["ids"]
        fingerprint = _metadata_fingerprint(source, grouped[source]["texts"])
        updated_metadatas = []

        for meta in grouped[source]["metadatas"]:
            updated = dict(meta)
            for key, value in metadata.items():
                if value:
                    updated[key] = value
            updated["metadata_fingerprint"] = fingerprint
            updated_metadatas.append(updated)

        try:
            collection.update(
                ids=source_ids,
                metadatas=updated_metadatas,
            )
        except Exception as e:
            print(f"Warning: could not save metadata for {source}: {e}")

    print(
        f"Metadata summary: {reused_count} reused, "
        f"{generated_count} generated."
    )

    if generated_count == 0:
        print("No LLM metadata generation was required.")

    return generated

## Build MultiVector Index

In [ ]:
def _document_metadata_text(meta: dict) -> str:
    """Compact text representation of a chunk's document-level metadata."""
    parts = [
        meta.get("document_type", ""),
        meta.get("document_category", ""),
        meta.get("location", ""),
        f"entities: {meta.get('entities', '')}",
        f"topics: {meta.get('topics', '')}",
    ]
    return " | ".join(p for p in parts if p)

In [ ]:
def build_multivector_indexes(
    client,
    collection,
    embedder,
    summary_collection_name: str,
    metadata_collection_name: str,
    batch_size: int = 64,
):
    """
    Build (or refresh) two auxiliary Chroma collections derived from the
    document-level metadata already stored on each chunk of `collection`.

    summary collection: one vector per chunk, embedding the document summary.
    metadata collection: one vector per chunk, embedding compact document
    metadata such as type, category, location, entities, and topics.

    Both collections use the same chunk IDs as `collection`, so fused results
    can be traced back to original chunks.
    """
    all_items = collection.get(include=["documents", "metadatas"])
    ids = all_items.get("ids", [])
    documents = all_items.get("documents", [])
    metadatas = all_items.get("metadatas", [])

    summary_collection = client.get_or_create_collection(
        summary_collection_name
    )
    metadata_collection = client.get_or_create_collection(
        metadata_collection_name
    )

    if not ids:
        return summary_collection, metadata_collection

    summary_texts = [
        (meta or {}).get("summary", "") or doc
        for doc, meta in zip(documents, metadatas)
    ]
    metadata_texts = [
        _document_metadata_text(meta or {})
        for meta in metadatas
    ]

    for start in range(0, len(ids), batch_size):
        end = start + batch_size
        batch_ids = ids[start:end]
        batch_metas = metadatas[start:end]
        batch_summary_texts = summary_texts[start:end]
        batch_metadata_texts = metadata_texts[start:end]

        summary_embeddings = embedder.encode(
            batch_summary_texts
        ).tolist()
        metadata_embeddings = embedder.encode(
            batch_metadata_texts
        ).tolist()

        summary_collection.upsert(
            ids=batch_ids,
            embeddings=summary_embeddings,
            documents=batch_summary_texts,
            metadatas=batch_metas,
        )

        metadata_collection.upsert(
            ids=batch_ids,
            embeddings=metadata_embeddings,
            documents=batch_metadata_texts,
            metadatas=batch_metas,
        )

    return summary_collection, metadata_collection

In [ ]:
summary_collection: str = "documents_summary_vectors"
metadata_collection: str = "documents_metadata_vectors"

(
    summary_vector_collection,
    metadata_vector_collection,
) = build_multivector_indexes(
    client,
    collection,
    embedder,
    summary_collection,
    metadata_collection,
)

## Chatbot Nodes — Intent Identification

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

INTENT_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are an intent classifier for a company documentation chatbot.

The chatbot has access to the following documents:
{documents}

Classify the user's CURRENT question into exactly one category.

1. document_question
The user is asking something that could potentially be answered using ANY
of the available company documents.

Examples:
- "What are the holidays?"
- "Give me the list of holidays in Bangalore"
- "What is the maternity leave policy?"
- "What does the IT policy say?"

2. document_list
The user wants to know what documents or source files are available.

Examples:
- "What documents do you have?"
- "Show me the available files."
- "What PDFs are available?"

3. document_filter
The user wants to search, restrict, or filter the chatbot to a particular
document or group of documents.

Examples:
- "Search only the holiday calendar."
- "Use the HARM document."
- "Only search the Bangalore holiday calendar."

4. out_of_scope
The question is clearly unrelated to ALL available documents.

Examples:
- "Write me a Python game."
- "What is the weather?"
- "Who is the president of India?"

IMPORTANT:
- Use available document names when deciding relevance.
- Use conversation history only to understand genuine follow-up references.
- Do NOT answer the question.
- Only classify the intent.
- Return one category.
- Confidence must be between 0 and 1.

Conversation history:
{history}

Current question:
{question}
""",
    ),
    ("human", "{question}"),
])

from typing import Literal
from pydantic import BaseModel, Field

class QueryIntent(BaseModel):
    """Structured output from the intent classifier."""

    intent: Literal[
        "document_question",
        "document_list",
        "document_filter",
        "out_of_scope",
    ] = Field(description="The intent of the user's question.")

    confidence: float = Field(
        description="Confidence between 0 and 1."
    )

In [ ]:
from langchain_groq import ChatGroq
import os
try:
    from google.colab import userdata
    key = userdata.get("GROQ_API_KEY")
except Exception:
    key = os.environ.get("GROQ_API_KEY")

groq_model = "openai/gpt-oss-20b"

llm = ChatGroq(
    model=groq_model,
    api_key=key,
    temperature=0,
    max_tokens=1200,
)

intent_chain = INTENT_PROMPT | llm.with_structured_output(QueryIntent)

In [ ]:
def list_documents(collection) -> list:
    """Return unique source files from ChromaDB."""
    all_items = collection.get(include=["metadatas"])
    sources = sorted({
        m.get("source_file", "unknown")
        for m in all_items["metadatas"]
        if m
    })
    return sources


all_sources = list_documents(collection)

question = "Explain the loan benefits."

intent_result = intent_chain.invoke({
    "history": "No previous conversation.",
    "documents": "\\n".join(f"- {s}" for s in all_sources),
    "question": question,
})

intent_result

## Question Metadata Extraction

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

METADATA_EXTRACTION_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a metadata extraction assistant for a company documentation RAG system.

Your job is to identify metadata constraints from the USER'S ORIGINAL QUESTION.

IMPORTANT:
You must analyze the ORIGINAL question exactly as provided.
Do NOT use a rewritten search query.
Do NOT inherit metadata from previous conversation turns unless the user
explicitly refers to that metadata.

Available document metadata:
{metadata}

Available document names:
{documents}

Rules:
1. Extract ONLY metadata explicitly stated or strongly implied by the CURRENT question.
2. Do NOT inherit a location from the previous question.
3. Do NOT inherit a document name from the previous question.
4. Do NOT inherit a topic from the previous question.
5. Do NOT invent metadata fields.
6. Only use metadata keys that actually appear in the available metadata.
7. document_category: for a CURRENT question that clearly indicates a document domain,
   use the normalized category when supported:
   - holiday / holidays / holiday calendar -> document_category=holiday
   - policy / policy document -> document_category=policy
   - FAQ / frequently asked questions -> document_category=faq
   - procedure / process / how-to -> document_category=procedure
8. For source_file:
   - Only identify a source_file when the CURRENT question explicitly names or clearly
     refers to the document.
   - Do NOT infer an exact filename merely from topic, location, or general subject.
   - Example: "What are the Bangalore holidays?" should normally return
     document_category=holiday and location=Bangalore, not an invented source_file.
9. If no reliable metadata filter can be identified, return an empty filters object.
10. Prefer no filter over an incorrect filter.
11. Do not answer the user's question.
12. Do not perform semantic retrieval.
13. Confidence must be between 0 and 1.

Conversation history:
{history}

Original current question:
{question}
""",
    ),
    ("human", "{question}"),
])

from pydantic import BaseModel, Field

class MetadataExtraction(BaseModel):
    """Structured output from metadata extraction."""

    filters: dict[str, str] = Field(
        default_factory=dict,
        description="Metadata key-value filters extracted from the user's original question.",
    )
    confidence: float = Field(
        description="Confidence between 0 and 1."
    )

In [ ]:
from langchain_groq import ChatGroq
import os
try:
    from google.colab import userdata
    key = userdata.get("GROQ_API_KEY")
except Exception:
    key = os.environ.get("GROQ_API_KEY")

groq_model = "openai/gpt-oss-20b"

llm = ChatGroq(
    model=groq_model,
    api_key=key,
    temperature=0,
    max_tokens=1200,
)

metadata_chain = (
    METADATA_EXTRACTION_PROMPT
    | llm.with_structured_output(MetadataExtraction)
)

In [ ]:
def get_available_metadata(collection) -> dict:
    """Inspect metadata fields and their available values from the Chroma collection."""
    all_items = collection.get(include=["metadatas"])
    metadata_values: dict[str, set] = {}

    for meta in all_items.get("metadatas", []):
        if not meta:
            continue
        for key, value in meta.items():
            if value is None:
                continue
            metadata_values.setdefault(key, set()).add(str(value))

    return {
        key: sorted(values)
        for key, values in metadata_values.items()
    }


def format_available_metadata(
    available_metadata,
    max_values_per_field: int = 100,
) -> str:
    """Convert available metadata into a compact string for the extraction LLM."""
    if not available_metadata:
        return "No metadata is available."

    parts = []
    for key, values in available_metadata.items():
        display_values = values[:max_values_per_field]
        values_text = ", ".join(display_values)
        if len(values) > max_values_per_field:
            values_text += ", ..."
        parts.append(f"{key}: {values_text}")

    return "\\n".join(parts)


all_sources = list_documents(collection)
available_metadata = get_available_metadata(collection)

question = "What are the Bangalore holidays?"

metadata_result = metadata_chain.invoke({
    "history": "No previous conversation.",
    "documents": "\\n".join(f"- {s}" for s in all_sources),
    "metadata": format_available_metadata(available_metadata),
    "question": question,
})

metadata_result.filters

## Multi-vector retrieval + Reciprocal Rank Fusion

In [ ]:
from typing import TYPE_CHECKING, Any, Iterable

def _query_vector_store(
    store,
    query_embedding,
    candidate_k: int,
    where: dict | None = None,
) -> list[dict]:
    """
    Query a single Chroma collection and return a flat list of candidates
    ordered by relevance (Chroma returns results ordered by distance).
    """
    query_kwargs = {
        "query_embeddings": [query_embedding],
        "n_results": candidate_k,
        "include": ["documents", "metadatas", "distances"],
    }

    if where:
        query_kwargs["where"] = where

    try:
        results = store.query(**query_kwargs)
    except Exception:
        return []

    ids = (results.get("ids") or [[]])[0]
    documents = (results.get("documents") or [[]])[0]
    metadatas = (results.get("metadatas") or [[]])[0]
    distances = (results.get("distances") or [[]])[0] if results.get("distances") else [None] * len(ids)

    return [
        {
            "id": item_id,
            "document": doc,
            "metadata": meta or {},
            "distance": distance,
        }
        for item_id, doc, meta, distance in zip(ids, documents, metadatas, distances)
    ]


def _rrf(
    result_lists: Iterable[list[dict]],
    candidate_k: int,
    k: int = 60,
) -> list[dict]:
    """
    Reciprocal Rank Fusion across several ranked candidate lists.

    Each candidate contributes 1 / (k + rank) for every list it appears in.
    Candidates are deduplicated by id; the first-seen document/metadata is kept.
    """
    scores: dict[str, float] = {}
    items: dict[str, dict] = {}

    for result_list in result_lists:
        for rank, item in enumerate(result_list):
            item_id = item["id"]
            scores[item_id] = scores.get(item_id, 0.0) + 1.0 / (k + rank + 1)

            if item_id not in items:
                items[item_id] = item

    fused = sorted(
        items.values(),
        key=lambda item: scores[item["id"]],
        reverse=True,
    )
    return fused[:candidate_k]


def multivector_retrieve(
    content,
    summary,
    metadata,
    query_embedding,
    candidate_k: int = 20,
    where: dict | None = None,
):
    """Retrieve from content, summary, and metadata vector spaces, then fuse with RRF."""
    content_r = _query_vector_store(content, query_embedding, candidate_k, where)
    summary_r = _query_vector_store(summary, query_embedding, candidate_k, where)
    metadata_r = _query_vector_store(metadata, query_embedding, candidate_k, where)

    fused = _rrf(
        [content_r, summary_r, metadata_r],
        candidate_k=candidate_k,
    )

    return fused, content_r, summary_r, metadata_r

In [ ]:
query_embedding = embedder.encode([question]).tolist()[0]

fused, content_r, summary_r, metadata_r = multivector_retrieve(
    collection,
    summary_vector_collection,
    metadata_vector_collection,
    query_embedding,
    candidate_k=20,
)

print(
    f"Content candidates: {len(content_r)} | "
    f"Summary candidates: {len(summary_r)} | "
    f"Metadata candidates: {len(metadata_r)} | "
    f"Fused: {len(fused)}"
)

for item in fused[:5]:
    print(
        "-", item["metadata"].get("source_file"),
        "|", item["document"][:80].replace("\\n", " ")
    )

## Reranking

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

pairs = [(question, item["document"]) for item in fused]
scores = reranker.predict(pairs)

ranked = sorted(
    zip(scores, fused),
    key=lambda x: x[0],
    reverse=True,
)

for score, item in ranked[:5]:
    print(
        f"{score:.3f} | "
        f"{item['metadata'].get('source_file')} | "
        f"{item['document'][:70].replace(chr(10), ' ')}"
    )